In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

feature_cols = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
X = train[feature_cols].copy()
y = train["Exited"].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)

In [ ]:
def eval_pipeline(X_tr, X_v, y_tr, y_v, categorical_cols, numeric_cols):
    """Обучает Pipeline(OneHot+StandardScaler+LogReg) и возвращает ROC AUC на валидации."""
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ])
    model = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])
    model.fit(X_tr, y_tr)
    val_proba = model.predict_proba(X_v)[:, 1]
    return roc_auc_score(y_v, val_proba), model

results = {}

categorical_cols = ["Geography", "Gender"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
auc, _ = eval_pipeline(X_train, X_val, y_train, y_val, categorical_cols, numeric_cols)
results["baseline"] = auc
print(f"baseline: {auc:.4f}")

## Шаг 1. Исправить выброс в `EstimatedSalary`

В туториале 2 нашли 3 аномальные строки (15.75M, 6.09M, 1.48M) на фоне нормального
диапазона до ~200K. Клипируем по 99-му перцентилю, посчитанному **только на
`X_train`** (не на всём датасете — иначе test/val незаметно повлияют на порог).

In [ ]:
salary_cap = X_train["EstimatedSalary"].quantile(0.99)

X_train1 = X_train.copy()
X_val1 = X_val.copy()
X_train1["EstimatedSalary"] = X_train1["EstimatedSalary"].clip(upper=salary_cap)
X_val1["EstimatedSalary"] = X_val1["EstimatedSalary"].clip(upper=salary_cap)

auc, _ = eval_pipeline(X_train1, X_val1, y_train, y_val, categorical_cols, numeric_cols)
results["+ clip EstimatedSalary"] = auc
print(f"+ clip EstimatedSalary: {auc:.4f}")

**Результат: 0.8692 → 0.8695.** Эффект почти нулевой — 3 строки из 15000 (0.02%
данных) не смогли сильно сдвинуть `StandardScaler`. Полезный вывод: не все находки
из EDA дают заметный прирост score, и это нормально — важно было проверить, а не
просто предположить.

## Шаг 2. Проверить нелинейность `NumOfProducts`

In [ ]:
train.groupby("NumOfProducts")["Exited"].agg(["mean", "count"])

Результат оказался ещё резче, чем ожидалось из линейной корреляции (-0.29):

| NumOfProducts | churn rate | count |
|---|---|---|
| 1 | 38.0% | 6424 |
| 2 | 4.7% | 8339 |
| 3 | 95.6% | 206 |
| 4 | 100.0% | 31 |

Это не просто нелинейно — это почти детерминированная связь на краях (3-4 продукта
≈ гарантированный отток). Линейная модель с `NumOfProducts` как числом видит только
общий нисходящий тренд и полностью упускает этот провал на 2 и взлёт на 3-4.
Переводим признак в категориальный.

In [ ]:
categorical_cols2 = ["Geography", "Gender", "NumOfProducts"]
numeric_cols2 = [c for c in feature_cols if c not in categorical_cols2]

X_train2 = X_train1.copy()
X_val2 = X_val1.copy()

auc, _ = eval_pipeline(X_train2, X_val2, y_train, y_val, categorical_cols2, numeric_cols2)
results["+ NumOfProducts as categorical"] = auc
print(f"+ NumOfProducts as categorical: {auc:.4f}")

**Результат: 0.8695 → 0.9171.** Самый большой прирост во всём туториале (+0.048) —
ровно то, что и ожидалось: линейная модель наконец "увидела" форму зависимости,
а не только направление.

## Шаг 3. Новые признаки

In [ ]:
X_train3 = X_train2.copy()
X_val3 = X_val2.copy()
X_train3["IsBalanceZero"] = (X_train3["Balance"] == 0).astype(int)
X_val3["IsBalanceZero"] = (X_val3["Balance"] == 0).astype(int)
numeric_cols3 = numeric_cols2 + ["IsBalanceZero"]

auc, _ = eval_pipeline(X_train3, X_val3, y_train, y_val, categorical_cols2, numeric_cols3)
results["+ IsBalanceZero"] = auc
print(f"+ IsBalanceZero: {auc:.4f}")

**Результат: 0.9171 → 0.9168.** Небольшое ухудшение (в пределах шума) — флаг
нулевого баланса не добавил ничего сверх того, что уже видно из самого `Balance`
через `StandardScaler`. Гипотеза не подтвердилась, признак можно не включать в
финальный набор.

In [ ]:
X_train4 = X_train2.copy()
X_val4 = X_val2.copy()
bins = [0, 30, 40, 50, 60, 100]
X_train4["AgeGroup"] = pd.cut(X_train4["Age"], bins=bins, labels=False)
X_val4["AgeGroup"] = pd.cut(X_val4["Age"], bins=bins, labels=False)
categorical_cols4 = categorical_cols2 + ["AgeGroup"]

auc, _ = eval_pipeline(X_train4, X_val4, y_train, y_val, categorical_cols4, numeric_cols2)
results["+ AgeGroup"] = auc
print(f"+ AgeGroup: {auc:.4f}")

**Результат: 0.9171 → 0.9245** (считая от шага с `NumOfProducts`, без `IsBalanceZero`,
который решили не брать). Заметный прирост (+0.007) — возраст действительно влияет
на риск оттока ступенчато, не строго линейно.

In [ ]:
X_train5 = X_train4.copy()
X_val5 = X_val4.copy()
X_train5["BalanceSalaryRatio"] = X_train5["Balance"] / (X_train5["EstimatedSalary"] + 1)
X_val5["BalanceSalaryRatio"] = X_val5["Balance"] / (X_val5["EstimatedSalary"] + 1)
numeric_cols5 = numeric_cols2 + ["BalanceSalaryRatio"]

auc, _ = eval_pipeline(X_train5, X_val5, y_train, y_val, categorical_cols4, numeric_cols5)
results["+ BalanceSalaryRatio"] = auc
print(f"+ BalanceSalaryRatio: {auc:.4f}")

**Результат: 0.9245 → 0.9246.** Эффект практически нулевой — отношение баланса к
зарплате не несёт заметного сигнала сверх самих `Balance` и `EstimatedSalary` по
отдельности. Оставляем в наборе, т.к. не вредит, но не переоцениваем его роль.

## Шаг 4. Target encoding для `Geography`

Считаем средний `Exited` по каждой стране **строго на `X_train`** (после сплита),
чтобы не допустить утечку из `X_val`/`test` в кодирование.

In [ ]:
X_train6 = X_train5.copy()
X_val6 = X_val5.copy()

geo_target_mean = X_train6.join(y_train).groupby("Geography")["Exited"].mean()
print(geo_target_mean)

X_train6["Geography_te"] = X_train6["Geography"].map(geo_target_mean)
X_val6["Geography_te"] = X_val6["Geography"].map(geo_target_mean)

categorical_cols6 = ["Gender", "NumOfProducts", "AgeGroup"]
numeric_cols6 = numeric_cols5 + ["Geography_te"]

auc, final_model = eval_pipeline(X_train6, X_val6, y_train, y_val, categorical_cols6, numeric_cols6)
results["+ Geography target encoding"] = auc
print(f"+ Geography target encoding: {auc:.4f}")

**Результат: 0.9246 → 0.9246.** Без изменений — one-hot уже полностью улавливал
сигнал от `Geography` (всего 3 категории, one-hot с этим прекрасно справляется).
Target encoding обычно выигрывает у one-hot на признаках с **много** категорий
(десятки-сотни), а не с тремя — ожидаемый результат, а не ошибка.

## Шаг 5. Итоговая таблица

| Шаг | ROC AUC (val) |
|---|---|
| baseline (туториал 3) | 0.8692 |
| + клип EstimatedSalary | 0.8695 |
| + NumOfProducts как категория | **0.9171** |
| + IsBalanceZero | 0.9168 (хуже — не берём) |
| + AgeGroup | **0.9245** |
| + BalanceSalaryRatio | 0.9246 |
| + Geography target encoding | 0.9246 (без изменений — оставляем one-hot) |

Итоговый набор признаков: клип `EstimatedSalary`, `NumOfProducts` как категория,
`AgeGroup`, `BalanceSalaryRatio`, `Geography`/`Gender`/`NumOfProducts`/`AgeGroup`
через one-hot (без target encoding — не дал эффекта). `IsBalanceZero` не включён.

## Шаг 6. Итоговый сабмит

In [ ]:
test_fe = test[feature_cols].copy()
test_fe["EstimatedSalary"] = test_fe["EstimatedSalary"].clip(upper=salary_cap)
test_fe["AgeGroup"] = pd.cut(test_fe["Age"], bins=bins, labels=False)
test_fe["BalanceSalaryRatio"] = test_fe["Balance"] / (test_fe["EstimatedSalary"] + 1)
test_fe["Geography_te"] = test_fe["Geography"].map(geo_target_mean)

# финальная модель обучается на train + val (100% размеченных данных)
X_full = pd.concat([X_train6, X_val6])
y_full = pd.concat([y_train, y_val])

preprocessor_final = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols6),
    ("num", StandardScaler(), numeric_cols6),
])
final_pipeline = Pipeline([
    ("preprocess", preprocessor_final),
    ("classifier", LogisticRegression(max_iter=1000)),
])
final_pipeline.fit(X_full, y_full)

test_proba = final_pipeline.predict_proba(test_fe[categorical_cols6 + numeric_cols6])[:, 1]
submission = pd.DataFrame({"id": test["id"], "Exited": test_proba})
submission.to_csv("../submissions/features_logreg.csv", index=False)

## Итог туториала 4

```
Признаки: клип EstimatedSalary, NumOfProducts (категория), AgeGroup,
          BalanceSalaryRatio, Geography/Gender (one-hot)
Локальный ROC AUC (val): 0.9246 (было 0.8692 на baseline)
Kaggle public score:  0.93238 (было 0.89026)
Kaggle private score: 0.92871 (было 0.87486)
```

Главный вывод: почти весь прирост (+0.048 из +0.055 общего) дал один признак —
`NumOfProducts`, переведённый из числа в категорию после того, как EDA (шаг 2)
показал почти детерминированную, но совершенно нелинейную связь с оттоком
(4.7% при 2 продуктах против 95-100% при 3-4). Остальные признаки дали куда более
скромный эффект, а один (`IsBalanceZero`) не помог вовсе. Это типичная картина в
feature engineering: одна сильная, хорошо обоснованная находка обычно стоит больше,
чем десяток наугад придуманных признаков.

До топ-10 (≈0.933+) осталось совсем немного (0.92871 против 0.93344 у 10-го места)
— разрыв, который должен закрыть переход на градиентный бустинг в туториале 5.